# v2: subset 1-7
* extract label from reports 

# v3 : resume

In [1]:
import pydicom
from glob import glob
import pandas as pd 
import re
import numpy as np 
import matplotlib.pyplot as plt


In [2]:
train = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv')
true_labels_df = train[train.notna().all(axis=1)]


In [3]:
train_series = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv')
train_series

,StudyInstanceUID,SeriesInstanceUID,Fluid_Sensitive,Fat_Suppression,Anatomical_Plane
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.12343110195036213483...,1,1,Sagittal
1,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.13821229744997220641...,1,1,Axial
2,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.23084836536722595275...,0,0,Coronal
3,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.40734206102458723096...,1,1,Coronal
4,1.2.826.0.1.3680043.8.498.10004873229099053869...,1.2.826.0.1.3680043.8.498.75714899997203615784...,0,0,Sagittal
...,...,...,...,...,...
24366,1.2.826.0.1.3680043.8.498.99957611207313098696...,1.2.826.0.1.3680043.8.498.54309001783901668483...,1,1,Axial
24367,1.2.826.0.1.3680043.8.498.99957611207313098696...,1.2.826.0.1.3680043.8.498.62870298129639401344...,1,1,Coronal
24368,1.2.826.0.1.3680043.8.498.99957611207313098696...,1.2.826.0.1.3680043.8.498.75079892939579407106...,0,0,Coronal
24369,1.2.826.0.1.3680043.8.498.99957611207313098696...,1.2.826.0.1.3680043.8.498.75317509519712148912...,1,1,Sagittal


In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4407 entries, 0 to 4406
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   StudyInstanceUID  4407 non-null   object 
 1   Report            4407 non-null   object 
 2   ACL               58 non-null     float64
 3   MCL               58 non-null     float64
 4   Medial Meniscus   58 non-null     float64
 5   Lateral Meniscus  58 non-null     float64
 6   Medial OA         58 non-null     float64
 7   Lateral OA        58 non-null     float64
 8   PF OA             58 non-null     float64
 9   Effusion          58 non-null     float64
 10  Synovitis         58 non-null     float64
 11  Baker's           58 non-null     float64
 12  Contusion         58 non-null     float64
 13  Fracture          58 non-null     float64
dtypes: float64(12), object(2)
memory usage: 482.1+ KB


In [5]:
df_final = pd.read_csv('/kaggle/input/datasets/malihealavi/knee-abnormally-dataset/labeld_df.csv')
# df_final.drop(columns=['Unnamed: 0'], inplace=True)
df_final.to_csv('labeld_df.csv', index=False)

df_merged = pd.read_csv('/kaggle/input/datasets/malihealavi/knee-abnormally-dataset/df_merged.csv')
# df_merged.drop(columns=['Unnamed: 0'], inplace=True)
df_merged.to_csv('df_merged.csv', index=False)

test_merged = pd.read_csv('/kaggle/input/datasets/malihealavi/knee-abnormally-dataset/test_merged.csv')
test_merged.to_csv('test_merged.csv', index=False)

test_metadata = pd.read_csv('/kaggle/input/datasets/malihealavi/knee-abnormally-dataset/test_metadata.csv')
test_metadata.to_csv('test_metadata.csv', index=False)

In [6]:
# import re

# idx = true_labels_df.index
# OFFICIAL_TARGETS = [
#     'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
#     'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture'
# ]
# for target in OFFICIAL_TARGETS:              # includes 'PF OA' too
#     name = re.sub(r"[^A-Za-z0-9]", "", target)   # 'PF OA' -> 'PFOA'

#     pred_raw   = df_final.iloc[idx][target]
#     pred_round = (pred_raw >= 0.5).astype(int)

#     sub = train.iloc[idx][['Report', target]].copy()
#     sub[f'{name}_pred_round'] = pred_round
#     sub[f'{name}_pred_raw']   = pred_raw
#     sub.to_csv(f'{name}.csv', index=False)

# Store Series_files Address 

In [ ]:
import os, glob, random, gc
import pydicom

root_dir = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series" 
series_dirs = glob.glob(os.path.join(root_dir, "*", "*"))
print('series_dirs created')
series_files = []  # one sorted file list per usable series
for i, sdir in enumerate(series_dirs):
    if not os.path.isdir(sdir):
        continue
    files = glob.glob(os.path.join(sdir, "*.dcm"))
    if len(files) < 3:
        continue  # need at least 3 slices to build a triple-stack
    # sort by InstanceNumber — header-only read, no pixel data, so this is fast
    files.sort(key=lambda f: int(pydicom.dcmread(f, stop_before_pixels=True).InstanceNumber))
    series_files.append(files)


import pickle
# --- SAVE ---
output_path = "series_files.pkl"
with open(output_path, "wb") as f:
    pickle.dump(series_files, f)
print(f"Saved {len(series_files)} series paths to {output_path}")

# --- LOAD (Use this in subsequent runs/cells) ---
with open(output_path, "rb") as f:
    series_files = pickle.load(f)

# Train Label Extraction From Report

### label extraction : one by one , selected targets

In [ ]:
import re
import os
import gc
import json
import math
import time
import traceback
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 1. SETTINGS  (only change things in this block)
# ============================================================
TRAIN_CSV   = '/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv'
OUT_ROOT    = '/kaggle/input/datasets/malihealavi/knee-abnormally-dataset'
FINAL_CSV   = '/kaggle/working/labeled_df_by_group.csv'  # merged result (written only when ALL chunks are done)

# --- WHICH TARGETS TO RE-EXTRACT WITH THE NEW METHOD ---------------------------------

TARGETS_TO_RERUN = [ 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
                    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
OLD_LABELS_PATH = '/kaggle/input/datasets/malihealavi/knee-abnormally-dataset/labeld_df.csv'  

# Extra folders with finished chunks of THIS run (resume). If Kaggle restarts, save the
# chunk folder as a Kaggle Dataset and add its path here.
EXTRA_RESUME_DIRS = []

CHUNK_SIZE       = 100     # reports per chunk (smaller = less work lost if the session dies)
BATCH_SIZE       = 8       # reports per LLM call (if it runs out of memory it halves itself automatically)
MAX_REPORT_CHARS = 8000    # very long reports are cut to this many characters (protects GPU memory)
DEFAULT_SCORE    = 0.1     # used when a target could not be read ("not mentioned")

# quick check BEFORE the slow model download: is the old labels file there?
if TARGETS_TO_RERUN is not None and not os.path.isfile(OLD_LABELS_PATH):
    raise FileNotFoundError(f"Old labels file not found: {OLD_LABELS_PATH}. "
                            f"Fix OLD_LABELS_PATH (or set TARGETS_TO_RERUN = None to redo all 12 targets).")

# ============================================================
# 2. LOGIN + MODEL
# ============================================================
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)
model.eval()
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.pad_token_id = tokenizer.eos_token_id

# stop tokens: normal end token + Llama-3 end-of-turn token
EOS_IDS = [tokenizer.eos_token_id]
_eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
if isinstance(_eot, int) and _eot >= 0 and _eot not in EOS_IDS:
    EOS_IDS.append(_eot)

# ============================================================
# 3. TARGETS + PROMPTS
# ============================================================
OFFICIAL_TARGETS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

# group name -> (targets scored in that call, rules text)
# To score ONE target per call, just give each target its own entry here.
TARGET_GROUPS = {
    'Synovitis': (["Synovitis"], """positive = at least moderate synovial thickening/hypertrophy/inflammation. Negative = trivial/minimal.
  * Non-English terms: recognize these as referring to synovitis/synovium even without a "synov-" root:
    Greek "υμενίτιδα" (synovitis) / "υμένας" (synovial membrane)
    Turkish "sinovit" / "sinovyal"
    Bulgarian "синовит" / "синовиум"
    Spanish "sinovitis" / "sinovial", French "synovite" / "synovial",
    German "Synovitis" / "Synovialitis" / "synovial", Dutch "synovitis" / "synoviaal"
  * If synovitis/synovial thickening is named but called "mild", "minimal", or "low-grade",
    score 0.3 as usual -- do not round up to 0.7 just because the word appears.
  * SYNOVITIS-ONLY FALLBACK: if synovitis/synovial is never mentioned by name in the report,
    but a MODERATE or LARGE joint effusion is described, score Synovitis as 0.4
    (possible/implied synovitis -- these two findings often occur together and are sometimes
    graded jointly in practice). If effusion is small/trace/absent, this fallback does not
    apply -- use the normal 0.1 "not mentioned" rule."""),

    'Effusion': (["Effusion"], """positive = moderate or large joint fluid. Negative = trace/small/physiologic amount.
  * NEGATION CHECK FIRST: look for "no effusion", "no significant effusion", "no evidence of
    effusion", "effusion is absent" (or the equivalent in other languages) directly next to the
    effusion mention. If found, score exactly 0.0 -- do not let the word "effusion" alone pull
    the score up.
  * Also count indirect descriptions as effusion evidence, not just the word "fluid/effusion":
    distension/swelling of the suprapatellar recess or bursa (e.g. Dutch "opzetting van
    suprapatellaire recessus", "distended suprapatellar bursa").
  * Non-English terms for reference: Spanish "derrame articular", Turkish "eklem içi sıvı artışı",
    German "Gelenkerguss", Greek "συλλογή υγρού", Dutch "gewrichtsvocht", Bulgarian "излив".
  * Note: "small/mild/minimal/trace effusion" wording is genuinely mixed in this dataset's true
    labels (goes both ways) -- keep this band around 0.3-0.4 rather than pushing it to an extreme."""),

    
    'PF OA': (["PF OA"], """positive = moderate-or-large area of high-grade cartilage loss in the patellofemoral compartment. Negative = mild/trace chondral thinning or chondromalacia.
  * Trochlea counts: findings on the "trochlea"/"trochlear" (femoral trochlea) are part of the
    patellofemoral joint, same as "patella"/"patellar" findings -- score both the same way.
  * Descriptive terms count too: "full-thickness cartilage defect", "osteochondral defect",
    "chondral ulcer", "cartilage loss" in the patella or trochlea are direct evidence of cartilage
    damage there, even if the word "osteoarthritis"/"OA" is never used. Score by size/severity as
    normal (large/full-thickness = 0.7-1.0, small/focal = 0.3).
  * "Condropatía"/"chondropathy"/"chondromalacia" of the patella or trochlea, named WITHOUT any
    downgrading word (mild, trace, grade 1, low-grade) attached, should score 0.7, not 0.0 --
    only drop it to 0.3/0.0 when the report itself qualifies it as mild/trace/low-grade.
  * NEGATION CHECK: phrases like "without chondral lesion", "cartilage appears congruent",
    "intact patellar cartilage" mean score 0.0 -- do not let nearby words like "patellofemoral"
    or "trochlear" pull the score up."""),

    'ACL-MCL': (["ACL", "MCL"], """
  ACL-ONLY REFINEMENTS (apply ONLY to the ACL score -- do not change how 
  you score the MCL):
  * ACL: positive = high-grade partial or complete/full-thickness tear, rupture, fiber discontinuity. 
  Negative = mild signal change, degeneration, low-grade, thickening without discontinuity.
  * Know the ACL in every language (do not confuse it with the PCL, MCL, or LCL):
    Spanish "LCA" / "ligamento cruzado anterior" (note: LCM = medial collateral, LCP = posterior 
    cruciate -- NOT the ACL); German "VKB" / "vorderes Kreuzband" (HKB = posterior); Turkish "ön 
    çapraz bağ" / "anterior çapraz bağ" / "ACL"; Greek "πρόσθιος χιαστός σύνδεσµος"; Dutch 
    "voorste kruisband"; Bulgarian "предна кръстна връзка"; Croatian "prednji križni ligament".
  * Tear words that mean POSITIVE: rotura, ruptura, desgarro, Riss, Ruptur, Ausriss, rüptür, 
    yırtık, scheur, ruptuur, ρήξη, руптура, скъсване. "Complete" in other languages: komplet, 
    volledige, πλήρης, пълна, completa. Loss of continuity ("bütünlük kaybı", "devamsızlık", 
    "ασυνέχεια ινών", "subtotal/υφολική") also means positive. A tear known from history 
    ("anamnestisch bekannt") still counts if the report describes it.
  * Complete or full-thickness tear, rupture, or fiber discontinuity = 1.0.
  * Score the ACL only from sentences about the ACL. "PCL intact" / "LCP normal" / "arka çapraz 
    bağ normal" must NOT lower the ACL score, and MCL or LCL injuries must NOT raise it. If the 
    ACL is never mentioned, score 0.1.
  * Score 0.3 (below threshold): "interstitial", "intrasubstance", "grade 1", "grade 2", "sprain", 
    "low-grade partial tear", or signal change only -- unless discontinuity or a complete tear 
    is also stated.

  MCL-ONLY REFINEMENTS (apply ONLY to the MCL score -- do not change how 
  you score the ACL):
    * MCL: positive = high-grade partial or complete ACUTE tear. Negative = low-grade sprain, chronic/remote injury.
    * Scale: grade 1 / low-grade / mild sprain = 0.3. Grade 2 / partial tear = 0.7. Grade 3 / 
    complete tear / rupture / high-grade partial tear = 1.0.
    * Only the MEDIAL collateral ligament counts. Lateral collateral (LCL, "lateral kollateral", 
    "fibular collateral") and posteromedial oblique ligament findings must NOT raise the MCL 
    score. Spanish "LCM" = medial collateral ligament (NOT the ACL = LCA).
    * NORMAL = 0.0: "collateral ligaments normal / without alterations / within normal limits" 
    ("ligamentos colaterales sin alteraciones", "kollateral ligamanlar normal", "MCL intakt", 
    "both collateral ligaments normal") means the MCL is normal. Score exactly 0.0.
    * NOT MENTIONED = 0.1: if the report never mentions the MCL or collateral ligaments, score 0.1. 
    Do NOT guess an MCL injury because the ACL or a meniscus is injured.
    * Below threshold (0.3): thickening, thinning, periligamentous edema or fluid around the MCL, 
    or "enthesopathic" / "degenerative" / "chronic" change, with no tear or sprain grade stated.
    Fluid or edema around an otherwise intact MCL is not a tear. If the report's impression 
    names a partial tear or grade 2 injury, score 0.7. 
    """),

    'Medial Meniscus - Lateral Meniscus': (["Medial Meniscus", "Lateral Meniscus"], """- positive = signal/tear reaching the articular surface, OR a 
  clear morphologic tear (displaced fragment, bucket-handle, radial/flap tear). Negative = intrasubstance degeneration not reaching the surface.

  MEDIAL MENISCUS-ONLY REFINEMENTS (apply ONLY to the Medial Meniscus score -- do not change how 
  you score the Lateral Meniscus):
  * Score only the MEDIAL meniscus. A tear in the lateral meniscus must never raise the medial 
    score. Medial = "menisco interno" (Spanish), "Innenmeniskus" (German), "έσω µηνίσκος" (Greek), 
    "medial menisküs" (Turkish), "mediale meniscus" (Dutch), "медиален менискус" (Bulgarian). 
    Lateral = "menisco externo", "Außenmeniskus", "έξω µηνίσκος", etc.
  * EXPLICIT NORMAL = 0.0: if the medial meniscus is described as normal, intact, "not torn", 
    "no tear", "sin signos de rotura", "ohne Riss", or "conservado", score exactly 0.0. Do not 
    give 0.3 for this, and do not let tears elsewhere in the report change it. Use 0.3 only when 
    degeneration or intrasubstance signal is described without a tear.
  * "Consistent with" / "compatible with" / "in keeping with" / "ile uyumlu" a tear is a firm 
    statement, not hedged. Score it as a stated tear. Only "possible", "cannot exclude", 
    "questionable", "R/O", or "DD" count as hedged.
  * Tear words count in any language: tear, Riss, rotura, desgarro, yırtık, scheur, ρήξη, 
    руптура/скъсване. A named horizontal, radial, flap, or complex tear of the medial meniscus 
    scores 0.9-1.0, or 0.7 if the type is not given.

    LATERAL MENISCUS-ONLY REFINEMENTS (apply ONLY to the Lateral Meniscus score -- do not change 
  how you score the Medial Meniscus):
  * Score only the LATERAL meniscus. A tear in the medial meniscus must never raise the lateral 
    score. Lateral = "menisco externo/lateral" (Spanish), "Außenmeniskus" (German), "έξω µηνίσκος" 
    (Greek), "lateral menisküs" (Turkish), "laterale meniscus" (Dutch), "lateralni menisk" 
    (Croatian), "латерален менискус" (Bulgarian). If the report is split by compartment, a 
    meniscus sentence under the "lateral compartment" heading is the lateral meniscus.
  * EXPLICIT NORMAL = 0.0: "intact", "not torn", "no tear", "normal", "without signs of tear" 
    ("sin signos de rotura", "ohne Riss", "bez znakova rupture") for the lateral meniscus means 
    exactly 0.0, even if other structures in the report are torn. Mild degeneration with no tear 
    is 0.3.
  * HEDGED = 0.2-0.3: "suspicious", "suspected", "possible", "cannot exclude", "R/O", "DD" before 
    a lateral tear means it is not confirmed. Exception: if the FINDINGS state the tear 
    definitively and only the impression says "suspected", score it as a definite tear.
  * "Consistent with" / "compatible with" / "ile uyumlu" a tear is a firm statement, not hedged.
  * Below threshold (0.3): grade I/II signal, "meniscopathy", or "degeneration" with no tear 
    named, and signal described as NOT reaching the articular surface.
  * Words like "small" or "partial" describing a tear do not lower the score. A named radial, 
    horizontal, flap, complex, or bucket-handle tear scores 0.9-1.0 (0.7 if no type is given)."""),

    
    'Contusion-Fracture': (["Contusion", "Fracture"], """
      Contusion-ONLY REFINEMENTS (apply ONLY to the Contusion score -- do not change how you score the Fracture)
    - Contusion: positive = bone contusion / bone bruise / bone marrow edema pattern from trauma or impact.
      * Words that mean contusion in any language: bone contusion, bone bruise, bone marrow edema, 
        "contusión ósea", "edema óseo", "Knochenödem", "kemik iliği ödemi", "kontüzyon", "botoedeem", 
        "botcontusie", "костно-мозъчен едем", "контузионен", "koštani edem", Greek "οστεοµυελικό 
        οίδηµα", "οίδηµα του οστικού µυελού", "οστικός µώλωπας".
      * Any named bone contusion or marrow edema counts, even if described as mild, small, or 
        residual. Do NOT lower the score for those words. Score 0.7 if no extent is given, up to 1.0 
        if extensive or clearly traumatic.
      * A fracture elsewhere in the report does NOT cancel it. Score the marrow edema as its own 
        finding (for example "additional area of marrow edema without associated fracture").
      * "No bone contusion", "no marrow edema", "normal marrow signal" = 0.0. Not mentioned = 0.1.
      
        Fracture-ONLY REFINEMENTS (apply ONLY to the Fracture score -- do not change how you score the Contusion)
        - Fracture: positive = an ACUTE cortical break or fracture line. Negative = old/healed/chronic deformity.
      * Any named fracture counts, in any language: fracture, fractura, Fraktur, fraktür, fractuur, 
        фрактура, κάταγµα, prijelom -- including subchondral, impaction, and insufficiency types. A bony 
        avulsion also counts ("avulsion fracture", "knöcherner Ausriss"). Score 0.7 if no detail is 
        given, 0.9-1.0 if displaced, comminuted, or clearly acute.
      * Bone edema, marrow edema, bone contusion, or bone bruise WITHOUT any fracture word: score 0.3. 
        Not 0.1, and never higher than 0.3. ("contusiones óseas", "edema óseo", "kontüzyon", 
        "bone bruise", "οστικός µώλωπας", "botcontusie")
      * "No fracture" / "no fracture line" / "sin fractura" = 0.0. If bone contusion is also 
        described, score 0.3. If bone is not mentioned at all, score 0.1.
        """),
    
     'Medial OA - Lateral OA': (["Medial OA", "Lateral OA"], """
     * SCORE COMPARTMENTS STRICTLY SEPARATELY: 
       - Medial = medial femoral condyle (MFC), medial tibial plateau (MTP), medial compartment. 
       - Lateral = lateral femoral condyle (LFC), lateral tibial plateau (LTP), lateral compartment.
     * POSITIVE (0.75 - 1.0): Requires a MODERATE OR LARGE area of HIGH-GRADE cartilage loss, "full-thickness cartilage defect", "severe osteoarthritis", or "Grade IV chondromalacia" in that specific compartment. 
     * FOCAL DEFECTS (0.5): If the text explicitly limits a high-grade or full-thickness defect with words like "focal", "small", or "tiny" (e.g., "focal high-grade chondrosis"), do not score higher than 0.5.
     * MILD / LOW-GRADE (0.25): "Grade 1", "Grade 2", "low-grade chondrosis", "fissuring", "partial thickness loss", or "mild chondromalacia" (e.g., "Hondromalacija II", "condropatía leve") score 0.25. 
     * BARE "OA" / NO SEVERITY (0.5): General phrases like "medial and lateral OA", "tricompartmental chondrosis", or "artrosis" without specific high-grade/severe modifiers should score 0.5, not 0.75+.
     * NORMAL (0.0): "cartilage intact", "no focal chondrosis", or "preserved" in that specific compartment.
     * UNMENTIONED (0.1): If the compartment's cartilage is not mentioned, score 0.1.
     """),
    
    "Baker's": (["Baker's"], """
         * TARGET DEFINITION: A Baker's cyst is strictly a "popliteal cyst" or "Baker's cyst" in the popliteal fossa. 
         * EXCLUSIONS: DO NOT score subchondral cysts, parameniscal cysts, meniscal cysts, or ganglion cysts. If the text only mentions these, score 0.0.
         * POSITIVE (0.75 - 1.0): Requires a MODERATE or LARGE popliteal/Baker's cyst. A cyst described as "ruptured" or "complicated" (e.g., "scheur van popliteale cyste", "quiste roto") also scores 1.0.
         * NEGATIVE (0.0): If the cyst is described as "small", "tiny", "trace", "minimal", "pequeño", "gering", or "μικρή", score exactly 0.0. Trace and small cysts are considered physiologic and negative.
         * DIMENSIONS RULE: If exact dimensions are given without a size adjective, cysts with a maximum dimension under 3.5 cm (or 35 mm) should be considered SMALL and scored 0.0. 
         * UNMENTIONED (0.1): If no popliteal or Baker's cyst is mentioned, score 0.1.
         * NEGATION CHECK (0.0): "No Baker's cyst", "No popliteal cyst", "sin quiste poplíteo" = 0.0.
        """),
}


# safety check: every one of the 12 targets must be scored exactly once
_covered = sorted(t for targets, _ in TARGET_GROUPS.values() for t in targets)
assert _covered == sorted(OFFICIAL_TARGETS), f"TARGET_GROUPS does not cover the 12 targets exactly once: {_covered}"

# ---- which targets / groups are re-extracted in this run ----
if TARGETS_TO_RERUN is None:
    RUN_TARGETS = list(OFFICIAL_TARGETS)
else:
    _bad = [t for t in TARGETS_TO_RERUN if t not in OFFICIAL_TARGETS]
    assert not _bad, f"Unknown target name(s) in TARGETS_TO_RERUN: {_bad}"
    RUN_TARGETS = [t for t in OFFICIAL_TARGETS if t in TARGETS_TO_RERUN]   # official order, no duplicates
    assert RUN_TARGETS, "TARGETS_TO_RERUN is empty"
PARTIAL_MODE = len(RUN_TARGETS) < len(OFFICIAL_TARGETS)

# a different folder for every different selection, so chunks of different runs never get mixed
_slug = "all12" if not PARTIAL_MODE else "_".join(re.sub(r"[^a-z0-9]", "", t.lower()) for t in RUN_TARGETS)
OUT_DIR = os.path.join(OUT_ROOT, "llm_labels_" + _slug)
RESUME_DIRS = [OUT_DIR] + list(EXTRA_RESUME_DIRS)

BASE_PROMPT = """You are an expert MSK radiologist grading knee MRI reports against a specific severity-thresholded rubric (this matches how the official ground truth for this task was generated -- ambiguous or borderline findings are graded negative, not positive).

For each target listed below, output a float score reflecting how strongly the report supports a POSITIVE finding AT OR ABOVE the stated threshold:
    1.0 = clearly meets or exceeds the threshold (explicit high-grade / complete / moderate-or-large / acute, as defined for the target)
    0.7 = the pathology is stated but with no explicit grade/severity qualifier (e.g. a bare "tear", no "partial" or "complete" specified)
    0.3 = mentioned but EXPLICITLY described as below the threshold (mild, low-grade, trace, small, intrasubstance, chronic/old, degenerative)
    0.1 = not mentioned anywhere in the report
    0.0 = explicitly negated / normal / intact / unremarkable / no evidence of

Treat hedged or equivocal language ("possible", "cannot exclude", "questionable") as NOT meeting the threshold (score 0.2-0.3), per the rule that ambiguous/borderline findings are graded negative.

Reports may be in English, Turkish, Spanish, German, Greek, Dutch, French, or another language -- read and reason about them directly; never skip a report because you don't recognize the language. A statement like "no evidence of X", "X intact", or "X unremarkable" negates X regardless of what else is discussed in the report."""


def build_system_prompt(targets, rules):
    keys = json.dumps(targets, ensure_ascii=False)
    return (
        BASE_PROMPT
        + "\n\nTarget(s) to score now, with thresholds:\n" + rules
        + f"\n\nScore each target separately, using only the evidence about that target."
        + f"\nReturn ONLY a valid JSON dictionary with EXACTLY these {len(targets)} key(s) (spelled exactly as shown) "
          f"and float values -- no other text, no markdown fences, no explanation:\n{keys}"
        + "\nBefore responding, verify your JSON includes every key listed above."
    )


# build all system prompts once
GROUP_PROMPTS = {
    name: (targets, build_system_prompt(targets, rules))
    for name, (targets, rules) in TARGET_GROUPS.items()
    if any(t in RUN_TARGETS for t in targets)
}


def build_prompt(system_prompt, report):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Report:\n{report[:MAX_REPORT_CHARS]}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


# ============================================================
# 4. PARSING (never raises)
# ============================================================
def _normalize_key(k):
    return re.sub(r"[^a-z0-9]", "", str(k).lower())

_NORM_TO_OFFICIAL = {_normalize_key(t): t for t in OFFICIAL_TARGETS}
_NORM_TO_OFFICIAL.update({
    'bakerscyst': "Baker's", 'bakercyst': "Baker's", 'baker': "Baker's",
    'pfoa': 'PF OA', 'patellofemoraloa': 'PF OA',
})


def remap_keys(d):
    """Map any spelling/spacing variant of a target name back to the official name."""
    if not isinstance(d, dict):
        return {}
    out = {}
    for k, v in d.items():
        official = _NORM_TO_OFFICIAL.get(_normalize_key(k))
        if official is not None:
            out[official] = v
    return out


def extract_json(output):
    output = output.strip()
    try:
        return json.loads(output)
    except Exception:
        pass
    start, end = output.find('{'), output.rfind('}')
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(output[start:end + 1])
        except Exception:
            pass
    # last resort (e.g. cut-off JSON): pull out whatever "key": number pairs exist
    pairs = re.findall(r'"([^"]+)"\s*:\s*([0-9]*\.?[0-9]+)', output)
    if pairs:
        return {k: float(v) for k, v in pairs}
    return None


def to_score(v):
    try:
        x = float(v)
    except Exception:
        return None
    if x != x:           # NaN
        return None
    return min(max(x, 0.0), 1.0)


def parse_output(text, targets):
    """Return {target: score} for the targets that could be read. Never raises."""
    if not isinstance(text, str) or not text.strip():
        return {}
    try:
        parsed = remap_keys(extract_json(text))
    except Exception:
        parsed = {}
    out = {}
    for t in targets:
        if t in parsed:
            s = to_score(parsed[t])
            if s is not None:
                out[t] = s
    # single-target call where the model returned only a bare number
    if not out and len(targets) == 1:
        m = re.fullmatch(r'\s*([01](?:\.\d+)?)\s*', text)
        if m:
            out[targets[0]] = float(m.group(1))
    return out


# ============================================================
# 5. SAFE BATCH GENERATION (never raises)
# ============================================================
@torch.inference_mode()
def _generate(prompts, max_new_tokens):
    enc = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=EOS_IDS,
        pad_token_id=tokenizer.eos_token_id,
    )
    gen = out[:, enc["input_ids"].shape[1]:]
    return tokenizer.batch_decode(gen, skip_special_tokens=True)


def safe_generate(prompts, max_new_tokens):
    """Returns one string per prompt (None if that prompt failed).
    On ANY error (e.g. out of GPU memory) the batch is split in half and retried."""
    err_msg = None
    try:
        return _generate(prompts, max_new_tokens)
    except Exception as e:
        err_msg = f"{type(e).__name__}: {str(e)[:150]}"
    # cleanup happens OUTSIDE the except block so the memory can really be freed
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if len(prompts) == 1:
        print(f"    [skipped 1 report] {err_msg}")
        return [None]
    mid = len(prompts) // 2
    return safe_generate(prompts[:mid], max_new_tokens) + safe_generate(prompts[mid:], max_new_tokens)


# ============================================================
# 6. LABEL ONE CHUNK
# ============================================================
def process_chunk(chunk):
    reports = chunk['Report'].tolist()
    n = len(reports)
    valid = [i for i, r in enumerate(reports) if isinstance(r, str) and len(r.strip()) >= 10]
    order = sorted(valid, key=lambda i: len(reports[i]))      # similar lengths together = less padding
    scores = {t: [float('nan')] * n for t in RUN_TARGETS}

    for gname, (targets, system_prompt) in GROUP_PROMPTS.items():
        max_new = 30 * len(targets) + 20
        for b in range(0, len(order), BATCH_SIZE):
            idxs = order[b:b + BATCH_SIZE]
            prompts = [build_prompt(system_prompt, reports[i]) for i in idxs]
            outputs = safe_generate(prompts, max_new)
            for i, text in zip(idxs, outputs):
                for t, v in parse_output(text, targets).items():
                    if t in scores:                      # only keep the targets we are re-extracting
                        scores[t][i] = v
        print(f"    done: {gname}")

    new_df = pd.DataFrame(scores, index=chunk.index)
    new_df = new_df.apply(pd.to_numeric, errors='coerce')
    n_missing = new_df.isna().sum(axis=1)                     # re-run targets that could not be read

    if PARTIAL_MODE:
        # start from the OLD rows; targets that could not be read keep their old value
        base = OLD_ALL.loc[chunk.index].copy()
        new_df = new_df.fillna(base[RUN_TARGETS])
        new_df = new_df.fillna(DEFAULT_SCORE)
        base = base.drop(columns=['llm_n_missing'], errors='ignore')
        for t in RUN_TARGETS:
            base[t] = new_df[t]
    else:
        new_df = new_df.fillna(DEFAULT_SCORE)
        drop_cols = [c for c in OFFICIAL_TARGETS + ['llm_n_missing'] if c in chunk.columns]
        base = pd.concat([chunk.drop(columns=drop_cols), new_df[OFFICIAL_TARGETS]], axis=1)

    base['llm_n_missing'] = n_missing
    return base


# ============================================================
# 7. SAVE / RESUME HELPERS
# ============================================================
def save_atomic(df, path):
    """Write to a temp file first, then rename, so a crash can never leave a half-written file."""
    tmp = path + ".tmp"
    df.to_csv(tmp, index=True)
    os.replace(tmp, path)


def chunk_filename(i):
    return f"chunk_{i + 1:03d}.csv"


def find_saved_chunk(i):
    for d in RESUME_DIRS:
        p = os.path.join(d, chunk_filename(i))
        if os.path.exists(p):
            return p
    return None


def load_saved_chunk(path, expected_index):
    """Load a saved chunk; return None if it looks wrong (then it is recomputed)."""
    try:
        df = pd.read_csv(path, index_col=0)
    except Exception:
        return None
    if len(df) != len(expected_index) or not df.index.equals(expected_index):
        return None
    if not all(c in df.columns for c in OFFICIAL_TARGETS):
        return None
    if df[OFFICIAL_TARGETS].isna().any().any():
        return None
    return df


# ============================================================
# 8. MAIN LOOP  (run once -> all 12 targets, all chunks)
# ============================================================
train = pd.read_csv(TRAIN_CSV)
assert 'Report' in train.columns, "train.csv has no 'Report' column"
df_sample = train.copy()
# df_sample = true_labels_df.copy()

n_rows = len(df_sample)
n_chunks = math.ceil(n_rows / CHUNK_SIZE)
os.makedirs(OUT_DIR, exist_ok=True)


def load_old_labels():
    """Read the single old-labels csv and match it to train.csv row by row."""
    old = pd.read_csv(OLD_LABELS_PATH)#.iloc[true_labels_df.index] 
    first = str(old.columns[0])
    if first.startswith('Unnamed'):                     # file was saved with index=True
        old = old.set_index(old.columns[0])
        old.index.name = None
        how = "matched by the saved index column"
    else:                                               # file was saved with index=False
        if len(old) != n_rows:
            raise ValueError(f"{OLD_LABELS_PATH} has no index column and {len(old)} rows, but train.csv has "
                             f"{n_rows} rows, so rows cannot be matched. Save the old table with index=True.")
        old.index = df_sample.index
        how = "no index column, matched by row order"
    old = old[~old.index.duplicated(keep='first')].sort_index()

    keep_cols = [t for t in OFFICIAL_TARGETS if t not in RUN_TARGETS]
    miss_cols = [t for t in keep_cols if t not in old.columns]
    if miss_cols:
        raise ValueError(f"Old labels file is missing these columns: {miss_cols}")
    for t in RUN_TARGETS:                 # old column may not exist for a re-run target: that is fine
        if t not in old.columns:
            old[t] = DEFAULT_SCORE

    miss_rows = df_sample.index.difference(old.index)
    if len(miss_rows) > 0:
        raise ValueError(f"Old labels cover only {len(old)} rows but train.csv has {n_rows} "
                         f"({len(miss_rows)} rows missing, first ones: {list(miss_rows[:5])}).")

    # safety: the reports in the old file must be the same reports as in train.csv
    if 'Report' in old.columns:
        same = (old.loc[df_sample.index, 'Report'].fillna('').astype(str).values == df_sample['Report'].fillna('').astype(str).values).mean()
        if same < 0.95:
            raise ValueError(f"Only {same:.0%} of the reports in the old labels file match train.csv "
                             f"(rows are probably in a different order). Save the old table with index=True.")
        if same < 1.0:
            print(f"  note: {same:.1%} of old reports are identical to train.csv (small text differences are OK)")

    old[OFFICIAL_TARGETS] = old[OFFICIAL_TARGETS].apply(pd.to_numeric, errors='coerce').fillna(DEFAULT_SCORE)
    print(f"Old labels loaded: {len(old)} rows ({how})")
    return old


OLD_ALL = load_old_labels() if PARTIAL_MODE else None

print(f"{n_rows} reports | {n_chunks} chunks of {CHUNK_SIZE} | batch size {BATCH_SIZE}")
print(f"Re-extracting: {RUN_TARGETS}")
if PARTIAL_MODE:
    print(f"Keeping OLD values for: {[t for t in OFFICIAL_TARGETS if t not in RUN_TARGETS]}")
print(f"LLM calls per report: {len(GROUP_PROMPTS)} -> {list(GROUP_PROMPTS)}")
print(f"Chunks are saved in: {OUT_DIR}")

all_parts, failed_chunks = [], []

for i in range(n_chunks):
    start, end = i * CHUNK_SIZE, min((i + 1) * CHUNK_SIZE, n_rows)
    chunk = df_sample.iloc[start:end].copy()

    # ---- resume: skip chunks that are already finished ----
    saved_path = find_saved_chunk(i)
    if saved_path is not None:
        done = load_saved_chunk(saved_path, chunk.index)
        if done is not None:
            print(f"[{i + 1}/{n_chunks}] already done, skipping ({saved_path})")
            all_parts.append(done)
            continue
        print(f"[{i + 1}/{n_chunks}] saved file looks wrong, recomputing")

    # ---- compute ----
    print(f"[{i + 1}/{n_chunks}] rows {start}-{end - 1} ...")
    t0 = time.time()
    try:
        chunk_final = process_chunk(chunk)
    except Exception:
        print(f"[{i + 1}/{n_chunks}] FAILED, will be retried on the next run:")
        traceback.print_exc()
        failed_chunks.append(i + 1)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        continue

    # ---- save right away ----
    try:
        save_atomic(chunk_final, os.path.join(OUT_DIR, chunk_filename(i)))
    except Exception:
        print(f"[{i + 1}/{n_chunks}] WARNING: could not save this chunk to disk:")
        traceback.print_exc()
    all_parts.append(chunk_final)

    dt = time.time() - t0
    left = n_chunks - i - 1
    print(f"[{i + 1}/{n_chunks}] saved | {dt / 60:.1f} min | reports with unread targets: "
          f"{int((chunk_final['llm_n_missing'] > 0).sum())} | rough time left: {dt * left / 60:.0f} min")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ============================================================
# 9. MERGE
# ============================================================
df_final = pd.concat(all_parts, axis=0).sort_index() if all_parts else None

if failed_chunks:
    print(f"\nChunks not finished: {failed_chunks}. Just run this cell again -- finished chunks are skipped.")
elif df_final is not None and len(df_final) == n_rows:
    save_atomic(df_final, FINAL_CSV)
    print(f"\nALL DONE. {len(df_final)} rows saved to {FINAL_CSV}")
    if PARTIAL_MODE:
        try:
            for t in RUN_TARGETS:
                old_v = OLD_ALL.loc[df_final.index, t]
                changed = int((old_v.round(3) != df_final[t].round(3)).sum())
                print(f"  {t}: {changed} of {len(df_final)} values changed vs old labels")
        except Exception as e:
            print("  (could not compare with old labels:", e, ")")
else:
    print("\nWARNING: row count does not match, check the chunk files in", OUT_DIR)

df_final

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
train = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv')
true_labels_df = train[train.notna().all(axis=1)]
# df_final = pd.read_csv('/kaggle/input/knee-abnormally-dataset/labeld_df.csv')
df_final = pd.read_csv('/kaggle/input/datasets/malihealavi/knee-abnormally-dataset/labeled_df_by_group.csv')
labeld_df = pd.read_csv('/kaggle/input/datasets/malihealavi/knee-abnormally-dataset/labeld_df.csv')
df_final [['Medial OA', 'Lateral OA']] = labeld_df[['Medial OA', 'Lateral OA']]
if 'Unnamed: 0' in df_final.columns:
    df_final.drop(columns=['Unnamed: 0'], inplace=True)
# 1. Define the 12 target classes exactly as they appear in the competition
TARGETS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 
    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture'
] # [cite: 148, 149, 150, 151, 152, 153, 154, 155, 156]

def calculate_macro_auc(df_true, df_pred, targets):
    auc_scores = {}
    
    for target in targets:
        # Extract the true binary labels and your predicted soft probabilities
        y_true = df_true[target].values
        y_pred = df_pred[target].values
        
        try:
            # Calculate AUC for this specific target
            auc = roc_auc_score(y_true, y_pred)
            auc_scores[target] = auc
        except ValueError:
            # Edge case: If the 58 samples only contain 0s (or only 1s) for a specific target,
            # roc_auc_score cannot be calculated and throws a ValueError.
            print(f"Warning: Skipping {target} - only one class present in ground truth subset.")
            auc_scores[target] = np.nan
            
    # Calculate Macro AUC by averaging the valid target scores 
    macro_auc = np.nanmean(list(auc_scores.values()))
    
    return auc_scores, macro_auc

# --- Example Usage ---


df_final_sub = df_final.iloc[true_labels_df.index] 
scores, final_macro_auc = calculate_macro_auc(true_labels_df, df_final_sub, TARGETS)

print("--- AUC Scores per Target ---")
for target, score in scores.items():
    if not np.isnan(score):
        print(f"{target}: {score:.4f}")

print("-" * 30)
print(f"Final Macro AUC: {final_macro_auc:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
train = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv')
true_labels_df = train[train.notna().all(axis=1)]
df_final = pd.read_csv('/kaggle/input/datasets/malihealavi/knee-abnormally-dataset/labeld_df.csv')
# df_final = pd.read_csv('/kaggle/working/labeled_df_by_group.csv')
if 'Unnamed: 0' in df_final.columns:
    df_final.drop(columns=['Unnamed: 0'], inplace=True)
# 1. Define the 12 target classes exactly as they appear in the competition
TARGETS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 
    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture'
] # [cite: 148, 149, 150, 151, 152, 153, 154, 155, 156]

def calculate_macro_auc(df_true, df_pred, targets):
    auc_scores = {}
    
    for target in targets:
        # Extract the true binary labels and your predicted soft probabilities
        y_true = df_true[target].values
        y_pred = df_pred[target].values
        
        try:
            # Calculate AUC for this specific target
            auc = roc_auc_score(y_true, y_pred)
            auc_scores[target] = auc
        except ValueError:
            # Edge case: If the 58 samples only contain 0s (or only 1s) for a specific target,
            # roc_auc_score cannot be calculated and throws a ValueError.
            print(f"Warning: Skipping {target} - only one class present in ground truth subset.")
            auc_scores[target] = np.nan
            
    # Calculate Macro AUC by averaging the valid target scores 
    macro_auc = np.nanmean(list(auc_scores.values()))
    
    return auc_scores, macro_auc

# --- Example Usage ---


df_final_sub = df_final.iloc[true_labels_df.index] 
scores, final_macro_auc = calculate_macro_auc(true_labels_df, df_final_sub, TARGETS)

print("--- AUC Scores per Target ---")
for target, score in scores.items():
    if not np.isnan(score):
        print(f"{target}: {score:.4f}")

print("-" * 30)
print(f"Final Macro AUC: {final_macro_auc:.4f}")

# Dicom Metadata Extraction

## Train Pre processsing

In [ ]:
import pydicom
import pandas as pd

def get_dicom_metadata_df(file_paths):
    """
    Read metadata from a list of DICOM file paths
    and return a sorted DataFrame.
    """
    rows = []

    for path in file_paths:
        ds = pydicom.dcmread(path, stop_before_pixels=True)

        row = {
            "file_path": path,
            # IDs (to link slices back to patient/study/series)
            "PatientID": getattr(ds, "PatientID", None),
            "StudyInstanceUID": getattr(ds, "StudyInstanceUID", None),
            "SeriesInstanceUID": getattr(ds, "SeriesInstanceUID", None),
            "SOPInstanceUID": getattr(ds, "SOPInstanceUID", None),

            # Order info (most important part)
            "SeriesNumber": getattr(ds, "SeriesNumber", None),
            "InstanceNumber": getattr(ds, "InstanceNumber", None),
            # "SliceLocation": getattr(ds, "SliceLocation", None),
            

            # Useful features for the model
            "SeriesDescription": getattr(ds, "SeriesDescription", None),
            "BodyPartExamined": getattr(ds, "BodyPartExamined", None),
            "Laterality": getattr(ds, "Laterality", None),
            "PatientSex": getattr(ds, "PatientSex", None),
            "ScanningSequence": getattr(ds, "ScanningSequence", None),
            # "SequenceVariant": getattr(ds, "SequenceVariant", None),
            # "ScanOptions": getattr(ds, "ScanOptions", None),
            "MRAcquisitionType": getattr(ds, "MRAcquisitionType", None),
            # "SliceThickness": getattr(ds, "SliceThickness", None),
            # "SpacingBetweenSlices": getattr(ds, "SpacingBetweenSlices", None),
            # "PixelSpacing": getattr(ds, "PixelSpacing", None),
            "Rows": getattr(ds, "Rows", None),
            "Columns": getattr(ds, "Columns", None),
            # "RepetitionTime": getattr(ds, "RepetitionTime", None),
            # "EchoTime": getattr(ds, "EchoTime", None),
            "MagneticFieldStrength": getattr(ds, "MagneticFieldStrength", None),
            "FlipAngle": getattr(ds, "FlipAngle", None),
            "Manufacturer": getattr(ds, "Manufacturer", None),
        }
        rows.append(row)

    df = pd.DataFrame(rows)

    # convert to numeric so sorting works correctly
    df["InstanceNumber"] = pd.to_numeric(df["InstanceNumber"], errors="coerce")

    # sort: group by series, then order slices inside each series
    df = df.sort_values(
        by=["SeriesInstanceUID", "InstanceNumber", ]
    ).reset_index(drop=True)

    return df

import glob

# --------------train
# file_paths = glob.glob("/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series/**/*.dcm", recursive=True)

# print(f"Found {len(file_paths)} dicom files")

# df = get_dicom_metadata_df(file_paths)
# df.head()
# df.to_csv('df_metadata.csv', index = False)



In [ ]:
import pandas as pd
train_series = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv')


group_keys = ["StudyInstanceUID", "SeriesInstanceUID"]
value_cols = [col for col in df.columns if col not in group_keys]

# 1. Identify columns that have at most 1 unique value per series group
constant_cols = [
    col
    for col in value_cols
    if df.groupby(group_keys)[col].nunique(dropna=False).max() <= 1
]

# 2. Build aggregation map dynamically
agg_dict = {
    col: ("first" if col in constant_cols else list) for col in value_cols
}

# 3. Aggregate and merge with the series dataset
dicom_grouped = df.groupby(group_keys, as_index=False).agg(agg_dict)
dicom_grouped

# 3. Merge grouped DICOM metadata into the series labels dataframe
df_merged = pd.merge(
    train_series,
    dicom_grouped,
    on=["StudyInstanceUID", "SeriesInstanceUID"],
    how="left",
)

import ast

# 1. Define standardization mappings
laterality_map = {"R": "RIGHT", "L": "LEFT" , }

manufacturer_map = {
    # Siemens
    "SIEMENS": "Siemens",
    "Siemens Healthineers": "Siemens",
    "Siemens": "Siemens",
    # Philips
    "Philips Medical Systems": "Philips",
    "Philips Healthcare": "Philips",
    "Philips": "Philips",
    # GE (Optional but recommended)
    "GE MEDICAL SYSTEMS": "GE",
    "GEHC": "GE",
    # Canon / Toshiba / Fujifilm / Hitachi (Optional cleanup)
    "CANON_MEC": "Canon",
    "TOSHIBA": "Toshiba",
    "FUJIFILM Healthcare Corporation": "Fujifilm",
    "Hitachi Medical Corporation": "Hitachi",
}

# 2. Apply replacements to DataFrame
df_merged["Laterality"] = df_merged["Laterality"].replace(laterality_map)
df_merged["Manufacturer"] = df_merged["Manufacturer"].replace(manufacturer_map)
cols = [ 'Laterality', 'Manufacturer' ]
for i in cols:
    print(i , ':', (df_merged[i].unique()))

df_merged.drop(columns= ['SOPInstanceUID','SeriesNumber' , 'PatientID'] , inplace=True)
df_merged['InstanceNumber'] = df_merged['InstanceNumber'].apply(lambda x: max(x))
df_merged
df_merged.to_csv('df_merged.csv' , index = False)

In [ ]:
import pandas as pd 
train_series = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv')
train_series

cols = ['Fluid_Sensitive','Fat_Suppression', 'Anatomical_Plane']
for i in cols:
    print(i , ':', (train_series[i].unique()))

gp = train_series.groupby('StudyInstanceUID').agg({
    'Anatomical_Plane':list,
    'SeriesInstanceUID' : 'count'
})
print(gp['SeriesInstanceUID'].unique())
gp[gp['SeriesInstanceUID']==3]
gp

## Test preprocessing

In [ ]:
import pydicom
import pandas as pd

def get_dicom_metadata_df(file_paths):
    """
    Read metadata from a list of DICOM file paths
    and return a sorted DataFrame.
    """
    rows = []

    for path in file_paths:
        ds = pydicom.dcmread(path, stop_before_pixels=True)

        row = {
            "file_path": path,
            # IDs (to link slices back to patient/study/series)
            "PatientID": getattr(ds, "PatientID", None),
            "StudyInstanceUID": getattr(ds, "StudyInstanceUID", None),
            "SeriesInstanceUID": getattr(ds, "SeriesInstanceUID", None),
            "SOPInstanceUID": getattr(ds, "SOPInstanceUID", None),

            # Order info (most important part)
            "SeriesNumber": getattr(ds, "SeriesNumber", None),
            "InstanceNumber": getattr(ds, "InstanceNumber", None),
            

            # Useful features for the model
            "SeriesDescription": getattr(ds, "SeriesDescription", None),
            "BodyPartExamined": getattr(ds, "BodyPartExamined", None),
            "Laterality": getattr(ds, "Laterality", None),
            "PatientSex": getattr(ds, "PatientSex", None),
            "ScanningSequence": getattr(ds, "ScanningSequence", None),
            "MRAcquisitionType": getattr(ds, "MRAcquisitionType", None),
            "Rows": getattr(ds, "Rows", None),
            "Columns": getattr(ds, "Columns", None),
            "MagneticFieldStrength": getattr(ds, "MagneticFieldStrength", None),
            "FlipAngle": getattr(ds, "FlipAngle", None),
            "Manufacturer": getattr(ds, "Manufacturer", None),
        }
        rows.append(row)

    df = pd.DataFrame(rows)

    # convert to numeric so sorting works correctly
    df["InstanceNumber"] = pd.to_numeric(df["InstanceNumber"], errors="coerce")

    # sort: group by series, then order slices inside each series
    df = df.sort_values(
        by=["SeriesInstanceUID", "InstanceNumber", ]
    ).reset_index(drop=True)

    return df

import glob

# ----------------test
file_paths = glob.glob("/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series/**/*.dcm", recursive=True)

print(f"Found {len(file_paths)} dicom files")

test_metadata = get_dicom_metadata_df(file_paths)
test_metadata.head()
test_metadata.to_csv('test_metadata.csv', index = False)
test_metadata

In [ ]:
import pandas as pd
import ast

test_series = pd.read_csv('/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series.csv')


group_keys = ["StudyInstanceUID", "SeriesInstanceUID"]
value_cols = [col for col in test_metadata.columns if col not in group_keys]

# 1. Identify columns that have at most 1 unique value per series group
constant_cols = [
    col
    for col in value_cols
    if test_metadata.groupby(group_keys)[col].nunique(dropna=False).max() <= 1
]

# 2. Build aggregation map dynamically
agg_dict = {
    col: ("first" if col in constant_cols else list) for col in value_cols
}

# 3. Aggregate and merge with the series dataset
dicom_grouped = test_metadata.groupby(group_keys, as_index=False).agg(agg_dict)
dicom_grouped

# 3. Merge grouped DICOM metadata into the series labels dataframe
df_merged = pd.merge(
    test_series,
    dicom_grouped,
    on=["StudyInstanceUID", "SeriesInstanceUID"],
    how="left",
)

# 1. Define standardization mappings
laterality_map = {"R": "RIGHT", "L": "LEFT" , }

manufacturer_map = {
    # Siemens
    "SIEMENS": "Siemens",
    "Siemens Healthineers": "Siemens",
    "Siemens": "Siemens",
    # Philips
    "Philips Medical Systems": "Philips",
    "Philips Healthcare": "Philips",
    "Philips": "Philips",
    # GE (Optional but recommended)
    "GE MEDICAL SYSTEMS": "GE",
    "GEHC": "GE",
    # Canon / Toshiba / Fujifilm / Hitachi (Optional cleanup)
    "CANON_MEC": "Canon",
    "TOSHIBA": "Toshiba",
    "FUJIFILM Healthcare Corporation": "Fujifilm",
    "Hitachi Medical Corporation": "Hitachi",
}

# 2. Apply replacements to DataFrame
df_merged["Laterality"] = df_merged["Laterality"].replace(laterality_map)
df_merged["Manufacturer"] = df_merged["Manufacturer"].replace(manufacturer_map)
cols = [ 'Laterality', 'Manufacturer' ]
for i in cols:
    print(i , ':', (df_merged[i].unique()))

df_merged.drop(columns= ['SOPInstanceUID','SeriesNumber' , 'PatientID'] , inplace=True)
df_merged['InstanceNumber'] = df_merged['InstanceNumber'].apply(lambda x: max(x))
df_merged
df_merged.to_csv('test_merged.csv' , index = False)
df_merged